# 06 Analysis

**Owner:** Part A each, Part B and C together

**What this notebook does:** answers the research question: how does democracy change before, during and after a conflict starts, and is it different for high and low arms importers?

**Reads:** `panel.csv`

**Writes:** `event_time.csv`, `event.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step.

## Setup

In [ ]:
import os
import pandas as pd
import plotly.express as px

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

In [ ]:
panel = pd.read_csv(CLEAN + "panel.csv")
print(panel.shape)

---
# Part A: Descriptives

## Nicolas: conflict by regime type

Paste from sandbox_nicolas, Part C. One sentence of result below each table.

In [ ]:
# your code here

In [ ]:
# your code here

## Yair: arms imports, democracy in war and peace

Paste from sandbox_yair, Part B. One sentence of result below each table.

In [ ]:
# your code here

In [ ]:
# your code here

In [ ]:
# your code here

---
# Part B: Event window (together)

The idea: we line up all conflicts as if they started in the same year, called `t = 0`.
Then we look at the average democracy index from 5 years before (`t = -5`) to 10 years after (`t = 10`).
Everything here uses dictionaries and loops, so you can check each step by printing.

**Step 1.** **Find the onsets.** An onset is a country-year with `conflict_active == 1` where the two years before had no conflict. First make a dictionary `status = {(iso3, year): conflict_active}` with one loop over `panel`. Then loop over `panel` again: a row is an onset if its `conflict_active` is 1 and `status.get((code, year - 1)) == 0` and `status.get((code, year - 2)) == 0`. Append `{"iso3": code, "onset_year": year}` to a list and make the dataframe `onsets`. Print how many onsets you found and the ones for Colombia and Ukraine.

In [ ]:
# your code here

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

`status.get(key)` returns `None` if the key does not exist. So years before 1950, and years right after a gap in the data,
can never count as "peaceful". That is what we want: a conflict already running in 1950 or 1951 cannot count as an onset.
Write that in the decisions table, and ask Justinas whether "two peaceful years" is a good definition.

</details>

**Step 2.** **Build the window.** Empty list `window`. Loop over `onsets`. For each onset, take the rows of that country with `year` between `onset_year - 5` and `onset_year + 10`. Loop over those rows and append a dictionary with `iso3`, `onset_year`, `year`, `t` (= `year - onset_year`), `democracy_index`, `arms_imports_tiv` and `max_intensity`. Make a dataframe `event`. Print the rows of one onset.

In [ ]:
# your code here

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

One country: `country_rows = panel[panel["iso3"] == code]`. The years around the onset: `country_rows[country_rows["year"].between(onset_year - 5, onset_year + 10)]`.

</details>

**Step 3.** **Change since the year before the conflict.** Make a dictionary `base = {}`. Loop over the rows of `event` with `t == -1`; if the democracy value is not missing (`not pd.isna(value)`), store `base[(iso3, onset_year)] = value`. Then loop over `event` again and build a list `changes`: if the key is in `base`, append `democracy_index - base[key]`, otherwise append `None`. Add it as the column `dem_change`. Print how many onsets have no base value. Why do we use the change and not the level? Write one sentence.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 4.** **High and low importers.** For each onset, compute the average `arms_imports_tiv` over `t = -5` to `t = -1`: loop over `event[event["t"] < 0]` and collect the values per onset in a dictionary of lists, then take the average of each list. Find the median of these averages. Above the median = `"high importer"`, otherwise `"low importer"`. Store it in a dictionary and add a column `importer_group` to `event` with a loop. Print the median, the share of onsets with an average of 0, and how many onsets are in each group.

In [ ]:
# your code here

In [ ]:
# your code here

<details><summary><b>Hint</b> (open only if stuck)</summary>

Many countries import nothing before a conflict, so the median may be 0. Then "above the median" simply means "imported anything".
That is fine, but write it in the decisions table ("the split is some imports vs none, groups are not equal in size").

</details>

**Step 5.** **Average per year.** Two small groupbys on `event`, both grouped by `["t", "importer_group"]`: the mean of `dem_change`, and `["dem_change"].count()` for the number of onsets with a value. Call them `means` and `counts`, reset the index of each, rename the count column to `n_countries` (it counts onsets, write that in a comment), and merge them: `event_time = pd.merge(means, counts, on=["t", "importer_group"])`. The columns must be `t`, `importer_group`, `dem_change`, `n_countries`.

In [ ]:
# your code here

In [ ]:
# your code here

**Step 6.** Save `event_time.csv` with `index=False`. Also save `event` as `event.csv`; you need it in Part C.

In [ ]:
# your code here

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
et = pd.read_csv(CLEAN + "event_time.csv")
assert list(et.columns) == ["t", "importer_group", "dem_change", "n_countries"]
assert et["t"].min() == -5 and et["t"].max() == 10
assert set(et["importer_group"].unique()) == {"high importer", "low importer"}
assert abs(et[et["t"] == -1]["dem_change"]).max() < 1e-9, "dem_change must be 0 at t = -1"
assert et["dem_change"].isna().sum() == 0
assert et["n_countries"].min() > 0
print(et[et["t"] == 0])
print("OK")

---
# Part C: Checks and findings (together)

**Step 1. Wars only.** Make a set of onset keys `(iso3, onset_year)` where `max_intensity == 2` in any year from `t = 0` to `t = 5`.
Filter `event` to those keys with a loop, keep the importer groups from the full sample, and redo only Step 5. Does the picture stay the same?

**Step 2. Group sizes.** How many onsets are in each group? Fewer than about 20 makes the lines unreliable. Say so.

**Step 3. Changing samples.** Plot `n_countries` by `t`. Then redo Step 5 with only onsets between 1955 and 2015 (full window),
and with only the first onset per country (onsets less than 10 years apart overlap). Do the lines change? Write your choice in the decisions table.

In [ ]:
# your code here

In [ ]:
# your code here

In [ ]:
# your code here

## Findings

Write three findings, one sentence each, with a number. These go into the app, the slides and the report.

1.
2.
3.

**What this cannot show:** (one sentence, for example that it shows a pattern, not a cause)

## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |